# B6 Stage2-A — 固定50時間構造のTP粗探索
既定の全セル実行は設定の表示のみです。ChatがStage2-A Freeze SHAを確認した後、Google Colabでのみ探索します。
1. 確認済み40桁SHAを `STAGE2A_FREEZE_SHA` へ設定し、`PREPARE_ENVIRONMENT=True` で準備します。
2. 入力用Driveをmountし、M1 rootとStage1候補・identity・effective configのパスを設定します。
3. Chat確認後のみ `CHAT_CONFIRMED_STAGE2A_FREEZE=True` と `RUN_STAGE2A_FULL=True` にします。
4. 全1,500条件は `/content/b6_stage2a` に保存されます。`stage2a_review.zip` をChatへ渡し、N06bを別途決定します。
候補や探索条件は変更できません。Stage2-B中心選定、Validation、Monitorの実行セルはありません。
`/content` はランタイム破棄で消えます。再開には出力フォルダ全体が必要です。Drive保存は明示ON時だけです。


In [ ]:
PREPARE_ENVIRONMENT = False
MOUNT_DRIVE = False
RUN_STAGE2A_FULL = False
CHAT_CONFIRMED_STAGE2A_FREEZE = False
SAVE_OUTPUT_TO_DRIVE = False
STAGE2A_FREEZE_SHA = ""
DATA_ROOT = "/content/drive/MyDrive/ゆうのすけさん2025"
CANDIDATE_PATH = "/content/drive/MyDrive/b6_stage1_20260929/stage1_selected_structures.json"
STAGE1_IDENTITY_PATH = "/content/drive/MyDrive/b6_stage1_20260929/identity.json"
STAGE1_EFFECTIVE_CONFIG_PATH = "/content/drive/MyDrive/b6_stage1_20260929/effective_config.json"
OUTPUT_ROOT = "/content/b6_stage2a"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/b6_stage2a_archive"
REPO_ROOT = "/content/b6_stage2a_repo"


In [ ]:
import json
FROZEN_CONFIG = json.loads('{"schema": "b6-stage2a-v1", "status": "STAGE2A_IMPLEMENTATION_FREEZE", "stage1_freeze_sha": "920b9be5c8f1bcf46a46dbbd4ac06dc19eb295b1", "candidate_sha256": "82a71c1ac9ffaa9a121795c6e0186bf77af5185b90bc21f65d7174161c35214f", "candidate_count": 50, "stage1_effective_config_sha256": "93cc9a92c6473cf770f98b72511de076375411db1c27ae1568f976f96212b37e", "discovery_start": "2020-01-01", "discovery_end_exclusive": "2024-01-01", "sl_pips": {"USDJPY": [10, 25, 40, 60, 95], "EURJPY": [15, 30, 50, 75, 115], "GBPJPY": [20, 40, 65, 100, 150], "AUDJPY": [10, 25, 40, 65, 100], "AUDUSD": [10, 20, 35, 55, 85], "EURAUD": [20, 35, 60, 95, 145], "GBPAUD": [20, 40, 70, 110, 160]}, "tp_ratios": ["0.5", "1.0", "1.5", "2.0", "3.0"], "include_tp_none": true, "tp_rounding": "Decimal ROUND_HALF_UP to 5 pips", "tp_min_pips": 5, "expected_configurations": 1500, "gate": {"min_trades": 150, "min_annual_trades": 30, "min_losses": 10, "min_pf": 1.1, "min_positive_years": 3}, "execution": {"timezone": "Europe/Helsinki->Asia/Tokyo naive", "ambiguous": "infer", "nonexistent": "shift_forward", "duplicates": "reject", "entry_exact": true, "entry_bar_inclusive": true, "exit_bar_inclusive": true, "same_bar": "SL_FIRST", "max_exit_fallback_minutes": 4, "ensure_exit_before_path": true, "hit_epsilon": 0, "interpolation": false, "gap_fill": "fixed_SL_price"}, "spread_pips": {"USDJPY": 0.5, "EURJPY": 1, "GBPJPY": 2, "AUDJPY": 1.5, "AUDUSD": 1.5, "EURAUD": 1.5, "GBPAUD": 2}, "metrics_basis": "unrounded_R", "pips_display_decimals": 6, "r_display_decimals": 9, "year_end_entry_stop": ["12-25", "01-03"], "early_gate_pruning": false, "stage2b_center_selection": false, "validation_enabled": false, "monitor_enabled": false, "output_policy": {"default_root": "/content/b6_stage2a", "drive_save_default": false, "all_1500_conditions": true, "all_trade_log_saved": false, "review_bundle": true}, "runtime": {"numpy": "2.3.5", "pandas": "2.2.3"}}')
print(json.dumps({"CandidateCount":50,"ExpectedConfigurations":1500,"Config":FROZEN_CONFIG},ensure_ascii=False,indent=2))


In [ ]:
if PREPARE_ENVIRONMENT:
    import importlib.util, subprocess, sys
    from pathlib import Path
    try:
        in_colab = importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        in_colab = False
    if not in_colab:
        raise PermissionError("Prepare in Google Colab only")
    if len(STAGE2A_FREEZE_SHA) != 40 or any(c not in "0123456789abcdef" for c in STAGE2A_FREEZE_SHA):
        raise ValueError("Chat確認済みStage2-A Freeze SHAを設定してください")
    subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","pandas==2.2.3"],check=True)
    repo = Path(REPO_ROOT)
    if not repo.exists():
        subprocess.run(["git","clone","https://github.com/TR-KJ/time-entry-portfolio-lab.git",str(repo)],check=True)
    subprocess.run(["git","diff","--quiet","HEAD","--"],cwd=repo,check=True)
    subprocess.run(["git","fetch","origin",STAGE2A_FREEZE_SHA],cwd=repo,check=True)
    subprocess.run(["git","checkout","--detach",STAGE2A_FREEZE_SHA],cwd=repo,check=True)
    sys.path.insert(0,str(repo/"src/research"))
    import numpy as np, pandas as pd
    if (np.__version__,pd.__version__) != ("2.3.5","2.2.3"):
        raise RuntimeError("固定ライブラリが既に読込済みの場合はランタイムを再起動し、準備セルを再実行してください")
    from b6.stage2a_search import verify_release
    verify_release(STAGE2A_FREEZE_SHA)
    from b6.stage2a_config import load_config
    if load_config() != FROZEN_CONFIG:
        raise ValueError("Notebook/config mismatch")
    print("Stage2-A preparation PASS")


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
if RUN_STAGE2A_FULL:
    if not PREPARE_ENVIRONMENT:
        raise PermissionError("固定コードの準備が必要です")
    from b6.stage2a_search import full_sweep
    summary = full_sweep(DATA_ROOT, OUTPUT_ROOT, CANDIDATE_PATH,
                         STAGE1_IDENTITY_PATH, STAGE1_EFFECTIVE_CONFIG_PATH,
                         STAGE2A_FREEZE_SHA, CHAT_CONFIRMED_STAGE2A_FREEZE)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
else:
    print("Stage2-A full sweep OFF。Chat確認後にColabで実行。")


In [ ]:
if RUN_STAGE2A_FULL and SAVE_OUTPUT_TO_DRIVE:
    import shutil
    from pathlib import Path
    destination = Path(DRIVE_OUTPUT_ROOT)
    if destination.exists():
        raise FileExistsError("退避先は新しいディレクトリを指定してください")
    shutil.copytree(OUTPUT_ROOT,destination)
    print("再開用の出力一式をDriveへ保存しました")
